In [1]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import librosa
import os
import json
from pathlib import Path
from sklearn.model_selection import train_test_split
import pandas as pd

2025-10-23 01:13:46.879947: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [11]:
class Config:
    # YAMNet parameters
    YAMNET_MODEL_HANDLE = 'https://tfhub.dev/google/yamnet/1'
    SAMPLE_RATE = 16000

    # Training parameters
    BATCH_SIZE = 32
    EPOCHS = 30
    LEARNING_RATE = 0.001

    # Sample limits for quick training
    TRAIN_SAMPLES = 10000
    VALIDATION_SAMPLES = 1000
    TEST_SAMPLES = 1000

    # Target instruments (common in worship music)
    TARGET_INSTRUMENTS = [
        'guitar',      # acoustic & electric
        'bass',        # bass guitar
        'keyboard',    # piano, organ, synth
        'string',      # violin, cello
        'brass',       # trumpet, trombone
        'reed',        # saxophone, flute
        'vocal'        # voice
    ]

    # Paths
    NSYNTH_DIR = './datasets/nsynth-train'  # Download and extract here
    SPLIT_DATA_DIR = f'./data/nsynth-{TRAIN_SAMPLES}/splits'  # Organized train/val/test folders
    MODEL_SAVE_DIR = './models/worship_flow'
    CHECKPOINT_DIR = './checkpoints'


In [12]:
class NSynthLoader:
    """Load and filter NSynth dataset"""

    def __init__(self, nsynth_path, target_instruments):
        self.nsynth_path = Path(nsynth_path)
        self.audio_dir = self.nsynth_path / 'audio'
        self.examples_path = self.nsynth_path / 'examples.json'
        self.target_instruments = target_instruments

        # Instrument family mapping
        self.family_mapping = {
            'guitar': ['guitar'],
            'bass': ['bass'],
            'keyboard': ['keyboard', 'mallet'],
            'string': ['string'],
            'brass': ['brass'],
            'reed': ['reed'],
            'vocal': ['vocal']
        }

    def load_metadata(self):
        """Load NSynth metadata"""
        print(f"Loading metadata from {self.examples_path}...")
        with open(self.examples_path, 'r') as f:
            metadata = json.load(f)
        return metadata

    def filter_samples(self, metadata, n_samples_per_class):
        """Filter samples by instrument family"""
        filtered = {inst: [] for inst in self.target_instruments}

        for note_id, note_data in metadata.items():
            family = note_data.get('instrument_family_str', '')

            # Map to target instrument
            for target, families in self.family_mapping.items():
                if family in families and target in self.target_instruments:
                    if len(filtered[target]) < n_samples_per_class:
                        filtered[target].append({
                            'note_id': note_id,
                            'family': family,
                            'pitch': note_data.get('pitch', 0),
                            'velocity': note_data.get('velocity', 0)
                        })
                    break

        return filtered

    def load_audio_samples(self, filtered_samples, sample_rate=16000, save_dir=None):
        """Load audio files for filtered samples and optionally save them"""
        X_data = []
        y_data = []
        file_paths = []

        label_to_idx = {inst: idx for idx, inst in enumerate(self.target_instruments)}

        for instrument, samples in filtered_samples.items():
            print(f"Loading {len(samples)} samples for {instrument}...")

            for sample in samples:
                audio_file = self.audio_dir / f"{sample['note_id']}.wav"

                if not audio_file.exists():
                    continue

                try:
                    # Load audio
                    waveform, sr = librosa.load(
                        str(audio_file),
                        sr=sample_rate,
                        duration=4.0  # NSynth notes are 4 seconds
                    )

                    # Ensure correct length (pad or truncate)
                    target_length = 4 * sample_rate
                    if len(waveform) < target_length:
                        waveform = np.pad(waveform, (0, target_length - len(waveform)))
                    else:
                        waveform = waveform[:target_length]

                    # Normalize
                    waveform = waveform / (np.max(np.abs(waveform)) + 1e-8)

                    X_data.append(waveform)
                    y_data.append(label_to_idx[instrument])
                    file_paths.append({
                        'note_id': sample['note_id'],
                        'instrument': instrument,
                        'original_path': str(audio_file)
                    })

                except Exception as e:
                    print(f"Error loading {audio_file}: {e}")

        return np.array(X_data, dtype=np.float32), np.array(y_data, dtype=np.int32), file_paths


In [13]:
# ============================================================================
# DATA PREPARATION
# ============================================================================

def save_split_data(X_data, y_data, file_paths, instruments, split_name, save_dir):
    """Save audio files to organized folders by split and instrument"""
    import soundfile as sf

    split_dir = Path(save_dir) / split_name

    # Create directories for each instrument
    for instrument in instruments:
        (split_dir / instrument).mkdir(parents=True, exist_ok=True)

    # Save each audio file
    saved_files = []
    for idx, (audio, label, file_info) in enumerate(zip(X_data, y_data, file_paths)):
        instrument = instruments[label]
        filename = f"{file_info['note_id']}.wav"
        file_path = split_dir / instrument / filename

        # Save using soundfile
        sf.write(file_path, audio, 16000)

        saved_files.append({
            'path': str(file_path),
            'instrument': instrument,
            'label': int(label),
            'note_id': file_info['note_id']
        })

    # Save metadata
    metadata_path = split_dir / 'metadata.json'
    with open(metadata_path, 'w') as f:
        json.dump({
            'split': split_name,
            'num_samples': len(saved_files),
            'instruments': instruments,
            'files': saved_files
        }, f, indent=2)

    print(f"Saved {len(saved_files)} files to {split_dir}")
    return saved_files

def prepare_nsynth_dataset(nsynth_path, config):
    """Prepare training, validation, and test datasets from NSynth"""

    loader = NSynthLoader(nsynth_path, config.TARGET_INSTRUMENTS)

    # Load metadata
    metadata = loader.load_metadata()
    print(f"Total samples in NSynth: {len(metadata)}")

    # Calculate samples per class
    total_samples = config.TRAIN_SAMPLES + config.VALIDATION_SAMPLES + config.TEST_SAMPLES
    samples_per_class = total_samples // len(config.TARGET_INSTRUMENTS)

    print(f"Targeting {samples_per_class} samples per instrument...")

    # Filter samples
    filtered = loader.filter_samples(metadata, samples_per_class)

    # Print distribution
    print("\nSample distribution:")
    for inst, samples in filtered.items():
        print(f"  {inst}: {len(samples)} samples")

    # Load audio data
    X_data, y_data, file_paths = loader.load_audio_samples(filtered, config.SAMPLE_RATE)

    print(f"\nTotal loaded samples: {len(X_data)}")

    # Split into train/val/test
    train_size = config.TRAIN_SAMPLES
    val_size = config.VALIDATION_SAMPLES
    test_size = config.TEST_SAMPLES

    # Ensure we don't exceed available data
    total_available = len(X_data)
    if total_available < train_size + val_size + test_size:
        print(f"Warning: Only {total_available} samples available.")
        # Adjust proportionally
        total_requested = train_size + val_size + test_size
        train_size = int(total_available * (train_size / total_requested))
        val_size = int(total_available * (val_size / total_requested))
        test_size = total_available - train_size - val_size

    # First split: separate test set
    X_temp, X_test, y_temp, y_test, paths_temp, paths_test = train_test_split(
        X_data, y_data, file_paths,
        test_size=test_size,
        stratify=y_data,
        random_state=42
    )

    # Second split: separate train and validation
    val_ratio = val_size / (train_size + val_size)
    X_train, X_val, y_train, y_val, paths_train, paths_val = train_test_split(
        X_temp, y_temp, paths_temp,
        test_size=val_ratio,
        stratify=y_temp,
        random_state=42
    )

    print(f"\nFinal split:")
    print(f"  Train: {len(X_train)} samples")
    print(f"  Validation: {len(X_val)} samples")
    print(f"  Test: {len(X_test)} samples")

    # Save organized data
    print(f"\nSaving organized data to {config.SPLIT_DATA_DIR}...")
    save_split_data(X_train, y_train, paths_train, config.TARGET_INSTRUMENTS, 'train', config.SPLIT_DATA_DIR)
    save_split_data(X_val, y_val, paths_val, config.TARGET_INSTRUMENTS, 'validation', config.SPLIT_DATA_DIR)
    save_split_data(X_test, y_test, paths_test, config.TARGET_INSTRUMENTS, 'test', config.SPLIT_DATA_DIR)

    return X_train, X_val, X_test, y_train, y_val, y_test


In [14]:
# Initialize config
config = Config()

print(f"Configuration:")
print(f"  Train samples: {config.TRAIN_SAMPLES}")
print(f"  Test samples: {config.TEST_SAMPLES}")
print(f"  Target instruments: {', '.join(config.TARGET_INSTRUMENTS)}")
print(f"  Batch size: {config.BATCH_SIZE}")
print(f"  Epochs: {config.EPOCHS}")
print(f"  Learning rate: {config.LEARNING_RATE}")
print()

Configuration:
  Train samples: 10000
  Test samples: 1000
  Target instruments: guitar, bass, keyboard, string, brass, reed, vocal
  Batch size: 32
  Epochs: 30
  Learning rate: 0.001



In [ ]:
# Check if NSynth exists
if not Path(config.NSYNTH_DIR).exists():
    print(f"ERROR: NSynth directory not found at {config.NSYNTH_DIR}")
    print("\nPlease download NSynth dataset:")
    print("1. Visit: https://magenta.tensorflow.org/datasets/nsynth")
    print("2. Download 'nsynth-train.jsonwav.tar.gz' (73GB)")
    print("3. Extract to ./nsynth-train/")
else:
    # Prepare dataset
    print("=" * 60)
    print("PREPARING DATASET")
    print("=" * 60)
    X_train, X_val, X_test, y_train, y_val, y_test = prepare_nsynth_dataset(
        config.NSYNTH_DIR,
        config
    )

PREPARING DATASET
Loading metadata from datasets/nsynth-train/examples.json...
Total samples in NSynth: 289205
Targeting 1714 samples per instrument...

Sample distribution:
  guitar: 1714 samples
  bass: 1714 samples
  keyboard: 1714 samples
  string: 1714 samples
  brass: 1714 samples
  reed: 1714 samples
  vocal: 1714 samples
Loading 1714 samples for guitar...
